# 🚚 Ejercicios: FedEx
# Flota y rutas: coste, consumo y emisiones

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de FedEx.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué tipo de vehículo
### cuesta más por
### entrega?

Coste total / entregas. El vehículo
caro puede salir barato por entrega.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_flota.csv'
)

g = (
  df
  .groupby('tipo')
  .agg(
    coste=('coste_eur', 'sum'),
    entregas=('entregas_dia',
              'sum'),
  )
)

g['coste_entrega'] = (
  g['coste'] / g['entregas']
).round(2)

print('Coste por entrega:')
print(
  g['coste_entrega']
  .sort_values(ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué vehículo emite
### más CO2 por km
### recorrido?

Emisiones / km normaliza por uso.
Clave para el objetivo de carbono.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_flota.csv'
)

g = (
  df
  .groupby('tipo')
  .agg(
    co2=('emisiones_co2', 'sum'),
    km=('km', 'sum'),
  )
)

g['g_co2_km'] = (
  g['co2'] / g['km'] * 1000
).round(1)

print('Gramos CO2 por km:')
print(
  g['g_co2_km']
  .sort_values(ascending=False)
)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuánto ahorran los
### eléctricos frente
### al diésel? (lift)

Comparamos coste por km de flota
limpia vs combustión.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_flota.csv'
)

limpio = ['Electrico', 'Bici-carga']
df['grupo'] = df['tipo'].apply(
  lambda t: 'Limpio'
  if t in limpio else 'Combustión'
)

df['coste_km'] = (
  df['coste_eur'] / df['km']
)

r = (
  df
  .groupby('grupo')['coste_km']
  .mean()
  .round(3)
)

dif = (
  r['Combustión'] / r['Limpio'] - 1
) * 100

print('Coste €/km por grupo:')
print(r)
print(f'\nCombustión +{dif:.0f}%')

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué rutas son muy
### largas y poco
### productivas?

Clasificamos km en tramos y vemos
las entregas medias de cada uno.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_flota.csv'
)

df['rango'] = pd.qcut(
  df['km'],
  q=4,
  labels=['Corta', 'Media',
          'Larga', 'Muy larga'],
)

r = (
  df
  .groupby('rango', observed=True)
  .agg(
    km=('km', 'mean'),
    entregas=('entregas_dia',
              'mean'),
  )
  .round(1)
)

print('Productividad por rango km:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué parte de la
### flota concentra las
### emisiones? (Pareto)

¿Pocos vehículos sucios cargan con
la mayoría del CO2 total?

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_flota.csv'
)

df['decil'] = pd.qcut(
  df['emisiones_co2'],
  q=[0, 0.8, 1.0],
  labels=['80% limpio',
          '20% sucio'],
)

r = (
  df
  .groupby('decil', observed=True)
  ['emisiones_co2']
  .sum()
)

pct = (r / r.sum() * 100).round(1)

print('CO2 total por grupo (kg):')
print(r.round(0))
print('\n% del CO2:')
print(pct)